# Pandas & Matplotlib
This part of the project uses Python (pandas and Matplotlib) to take the UK train rides data from raw CSV to analysis-ready tables.
- **Cleaning**
- **ETL** 
- **Analysis**

## Cleaning
The purpose of the cleaning process is to ensure that the data is valid and suitable for analysis.
<table style="margin-left:0">
<tr><th>Changes to Data</th>
<tr><td>Converted date columns into real dates</td>
<tr><td>Changed coulmn names to snake_case</td>
<tr><td>Replaced blanks with "No Delay" in reason_for_delay column</td>
<tr><td>Standardized Edinburgh station</td>
<tr><td>Standardized delay reasons</td>
<tr><td>None stays text</td>
</table>

In [217]:
import pandas as pd
import matplotlib.pyplot as plt
df = pd.read_csv('../data/raw/railway.csv', keep_default_na=False, na_values=[''])
df.info()
df.isna().sum()
df.duplicated().sum()

<class 'pandas.DataFrame'>
RangeIndex: 31653 entries, 0 to 31652
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   Transaction ID       31653 non-null  str  
 1   Date of Purchase     31653 non-null  str  
 2   Time of Purchase     31653 non-null  str  
 3   Purchase Type        31653 non-null  str  
 4   Payment Method       31653 non-null  str  
 5   Railcard             31653 non-null  str  
 6   Ticket Class         31653 non-null  str  
 7   Ticket Type          31653 non-null  str  
 8   Price                31653 non-null  int64
 9   Departure Station    31653 non-null  str  
 10  Arrival Destination  31653 non-null  str  
 11  Date of Journey      31653 non-null  str  
 12  Departure Time       31653 non-null  str  
 13  Arrival Time         31653 non-null  str  
 14  Actual Arrival Time  29773 non-null  str  
 15  Journey Status       31653 non-null  str  
 16  Reason for Delay     4172 non-nul

np.int64(0)

Changed column names to snake_case

In [218]:
df.columns = df.columns.str.lower().str.replace(" ", "_")
df = df.rename(columns={"date_of_journey": "journey_date", "arrival_time": "scheduled_arrival_time"})

Converted date columns into real dates

In [219]:
df["journey_date"] = pd.to_datetime(df["journey_date"])
df["date_of_purchase"] = pd.to_datetime(df["date_of_purchase"])

Replaced blanks with "No Delay" in reason_for_delay column

In [220]:
df["reason_for_delay"] = df["reason_for_delay"].fillna("No Delay")

Standardized Edinburgh station

In [221]:
# df.columns & applied .unique() to each column
# sorted(df["arrival_destination"].unique()) - found outliers
df["arrival_destination"] = df["arrival_destination"].replace({"Edinburgh": "Edinburgh Waverley"})

Standardized delay reasons

In [222]:
# df.columns & applied .value_counts() to each column
# df["reason_for_delay"].value_counts() - found outliers
df["reason_for_delay"] = df["reason_for_delay"].replace({
    "Signal failure": "Signal Failure",
    "Staff Shortage": "Staffing",
    "Weather Conditions": "Weather",
})

None stays text

In [223]:
# df = pd.read_csv('../data/raw/railway.csv', keep_default_na=False, na_values=[''])
# df.to_csv("../data/cleaned/railway.csv", index=False)

## ETL
The purpose of this ETL process is to create the fact & dimentions tables.
<table style="margin-left:0">
<tr><th>Step</th><th>Purpose</th></tr>
<tr><td>Extract</td><td>read the cleaned CSV</td></tr>
<tr><td>Transform</td><td>split it into dimensions and a fact table</td></tr>
<tr><td>Load</td><td>put the results into the warehouse</td></tr>
</table>

### dim_routes

In [224]:
dim_routes = df[["departure_station", "arrival_destination"]].drop_duplicates().rename(columns={"arrival_destination": "arrival_station"}).reset_index(drop=True).sort_values(["departure_station", "arrival_station"])
dim_routes["route_id"] = range(1, len(dim_routes) + 1)
dim_routes = dim_routes[["route_id", "departure_station", "arrival_station"]]
dim_routes.head(3)

,route_id,departure_station,arrival_station
52,1,Birmingham New Street,Coventry
47,2,Birmingham New Street,Edinburgh Waverley
41,3,Birmingham New Street,Liverpool Lime Street


### dim_railcard

In [225]:
dim_railcard = pd.DataFrame(sorted(df["railcard"].unique()), columns=["railcard_type"])
dim_railcard["railcard_id"] = range(1, len(dim_railcard) + 1)
dim_railcard = dim_railcard[["railcard_id", "railcard_type"]]
dim_railcard

,railcard_id,railcard_type
0,1,Adult
1,2,Disabled
2,3,None
3,4,Senior


### dim_sale_details

In [226]:
dim_sale_details = df[["purchase_type", "payment_method", "ticket_class", "ticket_type"]].drop_duplicates().sort_values(["purchase_type", "payment_method", "ticket_class", "ticket_type"]).reset_index(drop=True)
dim_sale_details["sale_details_id"] = range(1, len(dim_sale_details) + 1)
dim_sale_details = dim_sale_details[["sale_details_id", "purchase_type", "payment_method", "ticket_class", "ticket_type"]]
dim_sale_details.head(3)

,sale_details_id,purchase_type,payment_method,ticket_class,ticket_type
0,1,Online,Contactless,First Class,Advance
1,2,Online,Contactless,First Class,Anytime
2,3,Online,Contactless,First Class,Off-Peak


### dim_journey

In [227]:
df = df.rename(columns={"date_of_journey": "journey_date", "arrival_time": "scheduled_arrival_time"})

journey_lookup = (
    df[["journey_date", "departure_time", "scheduled_arrival_time", "actual_arrival_time", "journey_status", "reason_for_delay", "departure_station", "arrival_destination"]]
    .drop_duplicates()
    .sort_values(["journey_date", "departure_time", "departure_station", "arrival_destination"])
    .reset_index(drop=True)
)

journey_lookup["journey_id"] = range(1, len(journey_lookup) + 1)

dim_journey = journey_lookup[["journey_id", "journey_date", "departure_time", "scheduled_arrival_time", "actual_arrival_time", "journey_status", "reason_for_delay"]]
dim_journey.head(3)

,journey_id,journey_date,departure_time,scheduled_arrival_time,actual_arrival_time,journey_status,reason_for_delay
0,1,2024-01-01,00:00:00,00:30:00,00:30:00,On Time,No Delay
1,2,2024-01-01,00:00:00,01:50:00,02:07:00,Delayed,Signal Failure
2,3,2024-01-01,01:30:00,02:30:00,02:30:00,On Time,No Delay


### fact_ticket

In [228]:
fact_ticket = (
    df.merge(dim_railcard.rename(columns={"railcard_type": "railcard"}), on="railcard", how="left", validate="m:1")
    .merge(dim_routes.rename(columns={"arrival_station": "arrival_destination"}), on=["departure_station", "arrival_destination"], how="left", validate="m:1")
    .merge(dim_sale_details, on=["purchase_type", "payment_method", "ticket_class", "ticket_type"], how="left", validate="m:1")
    .merge(journey_lookup[["journey_date", "departure_time", "departure_station", "arrival_destination", "journey_id"]], on=["journey_date", "departure_time", "departure_station", "arrival_destination"], how="left", validate="m:1")
    .rename(columns={"date_of_purchase": "purchase_date", "time_of_purchase": "purchase_time"})
    [["transaction_id", "journey_id", "railcard_id", "route_id", "sale_details_id", "purchase_date", "purchase_time", "price", "refund_request"]]
)

fact_ticket.head(3)

# fact_ticket.to_csv("../data/star/fact_ticket.csv", index=False)
# dim_railcard.to_csv("../data/star/dim_railcard.csv", index=False)
# dim_routes.to_csv("../data/star/dim_routes.csv", index=False)
# dim_sale_details.to_csv("../data/star/dim_sale_details.csv", index=False)
# dim_journey.to_csv("../data/star/dim_journey.csv", index=False)

,transaction_id,journey_id,railcard_id,route_id,sale_details_id,purchase_date,purchase_time,price,refund_request
0,da8a6ba8-b3dc-4677-b176,34,1,32,4,2023-12-08,12:41:11,43,No
1,b0cdd1b0-f214-4197-be53,32,1,31,28,2023-12-16,11:23:01,23,No
2,f3ba7a96-f713-40d9-9629,152,3,23,10,2023-12-19,19:51:27,3,No


## Analysis
<table style="margin-left:0">
<tr><th>Insight</th><th>Chart</th></tr>
<tr><td>Service reliability</td><td>Bar</td></tr>
<tr><td>Worst routes</td><td>Horizontal bar</td></tr>
<tr><td>Delay reasons</td><td>Horizontal bar</td></tr>
<tr><td>Refund rate by status</td><td>Bar</td></tr>
<tr><td>Travel by hour</td><td>Line chart</td></tr>
<tr><td>How late delays are</td><td>Histogram</td></tr>
</table>

In [229]:
dim_journey = pd.read_csv("../data/star/dim_journey.csv")
dim_railcard = pd.read_csv("../data/star/dim_railcard.csv")
dim_routes = pd.read_csv("../data/star/dim_routes.csv")
dim_sale_details = pd.read_csv("../data/star/dim_sale_details.csv")
fact_ticket = pd.read_csv("../data/star/fact_ticket.csv")

Service reliability

In [230]:
df = pd.merge(fact_ticket, dim_journey, on="journey_id", how="left")
journey_status_counts = df["journey_status"].value_counts()

Worst routes

In [231]:
df = pd.merge(fact_ticket, dim_routes, on="route_id", how="left").merge(dim_journey, on="journey_id", how="left")
df["route"] = df["departure_station"] + " -> " + df["arrival_station"]
df = df.drop(columns=["departure_station", "arrival_station"])
pd.crosstab(df["route"], df["journey_status"], normalize="index").sort_values("Delayed", ascending=False).head(10)

journey_status,Cancelled,Delayed,On Time
route,,,
Edinburgh Waverley -> London Kings Cross,0.000000,1.000000,0.000000
London Euston -> York,0.000000,1.000000,0.000000
York -> Wakefield,0.000000,1.000000,0.000000
Liverpool Lime Street -> London Euston,0.090246,0.711030,0.198724
Manchester Piccadilly -> London Euston,0.008696,0.695652,0.295652
Liverpool Lime Street -> London Paddington,0.000000,0.481481,0.518519
Manchester Piccadilly -> Leeds,0.007042,0.450704,0.542254
Birmingham New Street -> Manchester Piccadilly,0.080357,0.428571,0.491071
Birmingham New Street -> London Euston,0.040000,0.352000,0.608000


Delay reasons

In [232]:
df = pd.merge(fact_ticket, dim_journey, on="journey_id", how="left")
filter = (df["reason_for_delay"] != "No Delay")
df[filter]["reason_for_delay"].value_counts()

reason_for_delay
Weather            1372
Signal Failure      970
Staffing            809
Technical Issue     707
Traffic             314
Name: count, dtype: int64

Refund rate by status

In [233]:
df = pd.merge(fact_ticket, dim_journey, on="journey_id", how="left")
df.groupby("journey_status")["refund_request"].apply(lambda s: (s == "Yes").mean())

journey_status
Cancelled    0.304255
Delayed      0.238220
On Time      0.000000
Name: refund_request, dtype: float64

Travel by hour

In [234]:
# df = pd.merge(fact_ticket, dim_journey, on="journey_id", how="left")
# pd.to_datetime(df["departure_time"], format="%H:%M:%S").dt.hour.value_counts().sort_index().head(3)

How late delays are

In [235]:
df = pd.merge(fact_ticket, dim_sale_details, on="sale_details_id", how="left")
df.groupby("purchase_type")["price"].agg(["count", "mean"])

,count,mean
purchase_type,,
Online,18521,20.665947
Station,13132,27.350518
